# Check the generated data
Before comparing models, inspect class counts, relation degrees, one injected ring, and the trips that make it a ring.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

ROOT = Path('..') if Path('../data/processed').exists() else Path('.')
DATA = ROOT / 'data' / 'processed'
riders = pd.read_parquet(DATA / 'riders.parquet')
drivers = pd.read_parquet(DATA / 'drivers.parquet')
trips = pd.read_parquet(DATA / 'trips.parquet')

In [ ]:
pd.Series({
    'riders': len(riders),
    'drivers': len(drivers),
    'trips': len(trips),
    'fraud_riders': int(riders.is_fraud.sum()),
    'rings': riders.ring_id.nunique(),
    'ring_trip_share': trips.is_ring_trip.mean(),
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for axis, column, title in zip(
    axes, ['driver_id', 'device_id', 'payment_id'], ['Drivers', 'Devices', 'Payments']
):
    degree = trips.groupby(column).rider_id.nunique()
    axis.hist(degree.clip(upper=degree.quantile(.99)), bins=30, color='#2878b5')
    axis.set(title=title, xlabel='Unique riders', ylabel='Entities')
plt.tight_layout()

In [ ]:
ring_id = riders.ring_id.dropna().iloc[0]
ring_trips = trips.loc[trips.ring_id == ring_id].copy()
graph = nx.Graph()
for row in ring_trips.itertuples():
    rider = f'r:{row.rider_id}'
    for kind, value in [('d', row.driver_id), ('dev', row.device_id), ('pay', row.payment_id)]:
        graph.add_edge(rider, f'{kind}:{value}')
colors = {'r': '#d1495b', 'd': '#2878b5', 'dev': '#edae49', 'pay': '#4c956c'}
node_colors = [colors[node.split(':', 1)[0]] for node in graph]
plt.figure(figsize=(12, 8))
nx.draw_networkx(graph, nx.spring_layout(graph, seed=7), node_color=node_colors, node_size=350, font_size=6)
plt.title(str(ring_id), loc='left', fontweight='bold')
plt.axis('off')

In [ ]:
ring_trips.sort_values(['rider_id', 'event_time'])[[
    'trip_id', 'rider_id', 'driver_id', 'device_id', 'payment_id',
    'fare', 'dist_m', 'hour', 'week'
]]